# P1-E16.7H-3 Renamed (P1-E11.H_3)FINAL — Regeneration, Failure-Control, Reviewer-Evidence, and Publication Lock

**Purpose.** Consolidate the final four-endpoint development evidence, automated-QA audit, age sensitivity, frozen external evaluation, probability quality, and authenticated dataset-shift results into one package.

**Scope.** This notebook performs no model fitting, selection, refitting, threshold optimization, recalibration, or domain adaptation. It regenerates tables and figures only from passed upstream locks.

**Evidence layers remain separate:** development out-of-fold evidence, retrospective locked official-test evidence, and locked external re-evaluation are never statistically pooled or silently ranked together.

## 1. Configuration and publication contract

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, shutil, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

SEED=20261012
ENDPOINTS=['normal_vs_ad','mci_vs_ad','normal_vs_mci','three_class']
ENDPOINT_LABELS={'normal_vs_ad':'Normal vs AD','mci_vs_ad':'MCI vs AD',
                 'normal_vs_mci':'Normal vs MCI','three_class':'Three class'}
CHANCE={'normal_vs_ad':.5,'mci_vs_ad':.5,'normal_vs_mci':.5,'three_class':1/3}
PROJECT_ROOT=Path(r'D:\EEG_AD_Research\dementia\paper one')
OUTPUTS_ROOT=PROJECT_ROOT/'paper1_outputs'
G_ROOT=OUTPUTS_ROOT/'p1_e16_7g_final_four_endpoint_publication_lock'
H1_ROOT=OUTPUTS_ROOT/'p1_e16_7h_1_exact_p1e07_automated_qa_sensitivity'
H2_ROOT=OUTPUTS_ROOT/'p1_e16_7h_2_external_compatibility_domain_shift_calibration_probability_quality'
H2A_ROOT=OUTPUTS_ROOT/'p1_e16_7h_2a_exact_artifact_resolution_supplement'
OUTPUT_ROOT=OUTPUTS_ROOT/'p1_e16_7h_3_final_regeneration_failure_control_reviewer_publication_lock'
TABLES=OUTPUT_ROOT/'tables'; FIGURES=OUTPUT_ROOT/'figures'; MANUSCRIPT=OUTPUT_ROOT/'manuscript_assets'
for folder in (OUTPUT_ROOT,TABLES,FIGURES,MANUSCRIPT): folder.mkdir(parents=True,exist_ok=True)

FILES={
 'G_integrity':G_ROOT/'tables'/'16_FINAL_INTEGRITY_CHECKS.csv',
 'G_development':G_ROOT/'tables'/'03_DEVELOPMENT_OOF_METHOD_RESULTS.csv',
 'G_official_test':G_ROOT/'tables'/'04_MATCHED_OFFICIAL_TEST_RESULTS.csv',
 'G_fairness':G_ROOT/'tables'/'06_CONTROLLED_FAIRNESS_RESULTS.csv',
 'G_age':G_ROOT/'tables'/'11_AGE_SENSITIVITY_RESULTS.csv',
 'G_age_deltas':G_ROOT/'tables'/'12_AGE_SENSITIVITY_PAIRED_DELTAS.csv',
 'G_age_distribution':G_ROOT/'tables'/'13_AGE_DISTRIBUTION_BY_DIAGNOSIS.csv',
 'H1_integrity':H1_ROOT/'tables'/'11_FINAL_INTEGRITY_CHECKS.csv',
 'H1_retention':H1_ROOT/'tables'/'04_RETENTION_BY_DIAGNOSIS.csv',
 'H1_omnibus':H1_ROOT/'tables'/'05_OMNIBUS_DIAGNOSIS_RETENTION_TEST.csv',
 'H1_pairwise':H1_ROOT/'tables'/'06_PAIRWISE_RETENTION_DIFFERENCES.csv',
 'H1_reasons':H1_ROOT/'tables'/'07_EXCLUSION_REASONS_BY_DIAGNOSIS.csv',
 'H1_thresholds':H1_ROOT/'tables'/'08_THRESHOLD_SCENARIO_RETENTION.csv',
 'H2_integrity':H2_ROOT/'tables'/'10_FINAL_INTEGRITY_CHECKS.csv',
 'H2_external':H2_ROOT/'tables'/'03_EXTERNAL_DISCRIMINATION_AND_PROBABILITY_QUALITY.csv',
 'H2_intervals':H2_ROOT/'tables'/'04_PARTICIPANT_STRATIFIED_BOOTSTRAP_INTERVALS.csv',
 'H2_calibration':H2_ROOT/'tables'/'05_CALIBRATION_BINS_WITH_WILSON_INTERVALS.csv',
 'H2A_integrity':H2A_ROOT/'tables'/'09_FINAL_INTEGRITY_CHECKS.csv',
 'H2A_internal_external':H2A_ROOT/'tables'/'05_INTERNAL_EXTERNAL_PROBABILITY_QUALITY_COMPARISON.csv',
 'H2A_shift':H2A_ROOT/'tables'/'06_FEATURE_SHIFT_STANDARDIZED_MEAN_DIFFERENCES.csv',
 'H2A_domain':H2A_ROOT/'tables'/'07_DOMAIN_CLASSIFIER_SEPARABILITY.csv',
 'H2A_provenance':H2A_ROOT/'tables'/'08_FINAL_INPUT_PROVENANCE.csv'}

def sha256_file(path,block=1<<20):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(block),b''): h.update(chunk)
    return h.hexdigest()
def save_table(q,name,manuscript=False):
    path=(MANUSCRIPT if manuscript else TABLES)/f'{name}.csv'; q.to_csv(path,index=False); return path
def save_figure(fig,name):
    fig.tight_layout()
    for root in (FIGURES,MANUSCRIPT):
        fig.savefig(root/f'{name}.pdf',bbox_inches='tight')
        fig.savefig(root/f'{name}.png',dpi=600,bbox_inches='tight',facecolor='white')
def pass_column(q):
    for c in ('pass','passed','authenticated'):
        if c in q.columns:return c
    raise KeyError(f'No pass column in {list(q.columns)}')
def bool_series(s): return s.astype(str).str.strip().str.lower().isin(['true','1','yes','pass','passed'])
sns.set_theme(style='whitegrid',context='paper',font_scale=1.08)
plt.rcParams.update({'pdf.fonttype':42,'ps.fonttype':42,'savefig.dpi':600})
print('Final publication output:',OUTPUT_ROOT)

## 2. Authenticate final upstream locks and stop on inconsistency

In [ ]:
missing=[str(p) for p in FILES.values() if not p.is_file()]
if missing: raise FileNotFoundError('Missing final evidence file(s):\n'+'\n'.join(missing))
inventory=pd.DataFrame([{'role':role,'path':str(path),'sha256':sha256_file(path),'size_bytes':path.stat().st_size} for role,path in FILES.items()])
save_table(inventory,'01_AUTHENTICATED_INPUT_INVENTORY')
locks=[]
for role in ('G_integrity','H1_integrity','H2_integrity','H2A_integrity'):
    q=pd.read_csv(FILES[role]); col=pass_column(q); passed=bool_series(q[col]).all()
    locks.append({'lock':role,'rows':len(q),'all_passed':bool(passed),'sha256':sha256_file(FILES[role])})
locks=pd.DataFrame(locks); save_table(locks,'02_UPSTREAM_LOCK_STATUS')
display(locks)
if not locks.all_passed.all(): raise AssertionError('At least one final upstream integrity lock is not fully passed.')
print('All final upstream evidence locks authenticated.')

## 3. Load and reconcile QA, development, age, external, and shift evidence

In [ ]:
retention=pd.read_csv(FILES['H1_retention']); omnibus=pd.read_csv(FILES['H1_omnibus'])
pairwise=pd.read_csv(FILES['H1_pairwise']); reasons=pd.read_csv(FILES['H1_reasons'])
thresholds=pd.read_csv(FILES['H1_thresholds']); development=pd.read_csv(FILES['G_development'])
official_test=pd.read_csv(FILES['G_official_test']); fairness=pd.read_csv(FILES['G_fairness'])
age=pd.read_csv(FILES['G_age']); age_deltas=pd.read_csv(FILES['G_age_deltas'])
age_distribution=pd.read_csv(FILES['G_age_distribution']); external=pd.read_csv(FILES['H2_external'])
intervals=pd.read_csv(FILES['H2_intervals']); calibration=pd.read_csv(FILES['H2_calibration'])
internal_external=pd.read_csv(FILES['H2A_internal_external']); shift=pd.read_csv(FILES['H2A_shift'])
domain=pd.read_csv(FILES['H2A_domain'])

checks=[
 {'check':'four development endpoints','pass':set(development.endpoint if 'endpoint' in development else development.task)==set(ENDPOINTS)},
 {'check':'four endpoints in internal-external table','pass':set(internal_external.task)>=set(ENDPOINTS)},
 {'check':'AHEPA restricted to Normal vs AD','pass':set(external.loc[external.dataset.eq('AHEPA'),'task'])=={'normal_vs_ad'}},
 {'check':'Dryad contains four endpoints','pass':set(external.loc[external.dataset.eq('Dryad'),'task'])==set(ENDPOINTS)},
 {'check':'QA denominator equals 979','pass':int(retention.montage_eligible_n.sum())==979},
 {'check':'QA retained total equals 914','pass':int(retention.retained_n.sum())==914},
 {'check':'feature shift covers AHEPA and Dryad','pass':set(shift.dataset)=={'AHEPA','Dryad'}},
 {'check':'domain audit covers AHEPA and Dryad','pass':set(domain.external_dataset)=={'AHEPA','Dryad'}}]
reconciliation=pd.DataFrame(checks); save_table(reconciliation,'03_EVIDENCE_RECONCILIATION'); display(reconciliation)
if not reconciliation['pass'].all(): raise AssertionError('Final evidence reconciliation failed.')

## 4. Manuscript Table 1 — Cohort retention and automated-QA sensitivity

In [ ]:
qa_table=retention[['diagnosis','montage_eligible_n','retained_n','rejected_n','retention_proportion']].copy()
qa_table['retention_percent']=100*qa_table.retention_proportion
qa_table['omnibus_chi2']=float(omnibus.chi2.iloc[0]); qa_table['omnibus_df']=int(omnibus.df.iloc[0])
qa_table['omnibus_p_value']=float(omnibus.p_value.iloc[0]); qa_table['cramers_v']=float(omnibus.cramers_v.iloc[0])
save_table(qa_table,'Table_1_Automated_QA_Retention',True)
display(qa_table.round(4))

threshold_table=thresholds.copy(); threshold_table['retention_percent']=100*threshold_table.retention_proportion
save_table(threshold_table,'Table_S1_QA_Threshold_Sensitivity',True)

## 5. Manuscript Table 2 — Development method comparison

In [ ]:
dev=internal_external.loc[internal_external.dataset.eq('CAUEEG development OOF')].copy()
dev['endpoint_label']=dev.task.map(ENDPOINT_LABELS)
dev=dev.sort_values(['task','balanced_accuracy'],ascending=[True,False])
dev['rank_within_endpoint']=dev.groupby('task').balanced_accuracy.rank(method='min',ascending=False).astype(int)
dev_table=dev[['endpoint_label','method','n','balanced_accuracy','roc_auc','multiclass_brier','prevalence_baseline_brier','brier_skill_score','rank_within_endpoint']]
save_table(dev_table,'Table_2_Development_OOF_Method_Comparison',True)
best_dev=dev.loc[dev.groupby('task').balanced_accuracy.idxmax(),['task','endpoint_label','method','n','balanced_accuracy','roc_auc','brier_skill_score']].sort_values('task')
save_table(best_dev,'Table_2A_Best_Development_Method_Per_Endpoint',True)
display(best_dev.round(4))

## 6. Manuscript Table 3 — Frozen external transportability and probability quality

In [ ]:
external_table=external.copy()
external_table['endpoint_label']=external_table.task.map(ENDPOINT_LABELS)
external_table=external_table[['dataset','endpoint_label','method','n','accuracy','balanced_accuracy','macro_f1','mcc','roc_auc',
                               'sensitivity','specificity','multiclass_brier','prevalence_baseline_brier','brier_skill_score','classwise_ece_10bin']]
save_table(external_table,'Table_3_Frozen_External_Transportability_and_Probability_Quality',True)
display(external_table.round(4))

shift_summary=shift.groupby('dataset',as_index=False).absolute_smd.agg(['median','mean','max']).reset_index()
shift_summary=shift_summary.merge(domain.rename(columns={'external_dataset':'dataset'}),on='dataset',how='left')
save_table(shift_summary,'Table_4_External_Feature_Shift_and_Domain_Separability',True)
display(shift_summary.round(4))

## 7. Manuscript Table 5 — Age sensitivity evidence

In [ ]:
# Preserve upstream column names: this table is a direct manuscript extract, not a re-analysis.
age_table=age.copy()
endpoint_col=next((c for c in ('endpoint','task') if c in age_table),None)
if endpoint_col: age_table['endpoint_label']=age_table[endpoint_col].map(ENDPOINT_LABELS)
save_table(age_table,'Table_5_Fold_Safe_Age_Sensitivity',True)
save_table(age_deltas,'Table_S2_Paired_Age_Sensitivity_Deltas',True)
save_table(age_distribution,'Table_S3_Age_Distribution_by_Diagnosis',True)
display(age_table.head(30))

## 8. Publication Figure 1 — QA retention and threshold stability

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4.8))
colors={'Normal':'#4C78A8','MCI':'#F2A541','AD':'#59A14F'}
q=qa_table.copy()
axes[0].bar(q.diagnosis,q.retention_percent,color=[colors[x] for x in q.diagnosis])
for i,v in enumerate(q.retention_percent): axes[0].text(i,v+1,f'{v:.1f}%',ha='center',fontsize=9)
axes[0].set_ylim(0,100); axes[0].set_ylabel('Retained recordings (%)'); axes[0].set_title('Locked automated-QA retention')

for diagnosis,z in threshold_table.groupby('diagnosis'):
    order=['stricter_10pct','locked','relaxed_10pct']; z=z.set_index('scenario').reindex(order)
    axes[1].plot(order,100*z.retention_proportion,marker='o',lw=2,label=diagnosis,color=colors[diagnosis])
axes[1].set_ylim(80,100); axes[1].set_ylabel('Retained recordings (%)'); axes[1].set_xlabel('QA threshold scenario')
axes[1].set_xticklabels(['10% stricter','Locked','10% relaxed']); axes[1].set_title('Threshold sensitivity'); axes[1].legend(frameon=False)
save_figure(fig,'Figure_1_QA_Retention_and_Threshold_Stability'); plt.show()

## 9. Publication Figure 2 — Four-endpoint development comparison

In [ ]:
plot=dev.copy(); endpoint_order=[ENDPOINT_LABELS[x] for x in ENDPOINTS]
fig,ax=plt.subplots(figsize=(10,6))
for method,z in plot.groupby('method'):
    z=z.set_index('endpoint_label').reindex(endpoint_order)
    ax.plot(endpoint_order,z.balanced_accuracy,marker='o',lw=1.8,label=method)
for endpoint in ENDPOINTS:
    ax.scatter(ENDPOINT_LABELS[endpoint],CHANCE[endpoint],marker='_',s=180,color='#B22222',zorder=5)
ax.set_ylim(.25,.9); ax.set_ylabel('Nested development OOF balanced accuracy')
ax.set_xlabel('Endpoint'); ax.set_title('Leakage-controlled development method comparison')
ax.legend(frameon=False,fontsize=7,ncol=2); save_figure(fig,'Figure_2_Development_Method_Comparison'); plt.show()

## 10. Publication Figure 3 — Internal-to-external transportability

In [ ]:
# Compare like evidence descriptively; do not pool or test across evidence layers.
conv_name='Conventional nested benchmark'
internal_conv=dev.loc[dev.method.eq(conv_name),['task','balanced_accuracy']].rename(columns={'balanced_accuracy':'CAUEEG development OOF'})
external_conv=external.loc[external.method.eq('Frozen conventional benchmark'),['dataset','task','balanced_accuracy']]
rows=[]
for r in external_conv.itertuples(index=False):
    iv=internal_conv.loc[internal_conv.task.eq(r.task),'CAUEEG development OOF']
    if len(iv): rows.extend([{'dataset':r.dataset,'task':r.task,'evidence':'CAUEEG development OOF','balanced_accuracy':float(iv.iloc[0])},
                             {'dataset':r.dataset,'task':r.task,'evidence':'Frozen external','balanced_accuracy':r.balanced_accuracy}])
transport=pd.DataFrame(rows); save_table(transport,'Table_S4_Internal_External_Descriptive_Transportability',True)
panels=sorted(transport.dataset.unique()); fig,axes=plt.subplots(1,len(panels),figsize=(11,5),sharey=True)
if len(panels)==1: axes=[axes]
for ax,dataset in zip(axes,panels):
    z=transport.loc[transport.dataset.eq(dataset)]
    for task,g in z.groupby('task'):
        g=g.set_index('evidence').reindex(['CAUEEG development OOF','Frozen external'])
        ax.plot([0,1],g.balanced_accuracy,marker='o',label=ENDPOINT_LABELS[task])
    ax.set_xticks([0,1]); ax.set_xticklabels(['Development\nOOF','Frozen\nexternal']); ax.set_title(dataset); ax.set_ylim(.25,.85)
axes[0].set_ylabel('Balanced accuracy'); axes[-1].legend(frameon=False,fontsize=7)
save_figure(fig,'Figure_3_Internal_to_External_Transportability'); plt.show()

## 11. Publication Figure 4 — Probability quality and dataset shift

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,5))
q=external.copy(); q['label']=q.apply(lambda r:f"{r.dataset} · {ENDPOINT_LABELS[r.task]} · {'Conv.' if r.method=='Frozen conventional benchmark' else 'Deep'}",axis=1)
colors_bar=['#4C78A8' if x=='Frozen conventional benchmark' else '#F28E2B' for x in q.method]
axes[0].barh(np.arange(len(q)),q.brier_skill_score,color=colors_bar); axes[0].axvline(0,color='.25',ls='--')
axes[0].set_yticks(np.arange(len(q))); axes[0].set_yticklabels(q.label,fontsize=7); axes[0].set_xlabel('Brier skill score'); axes[0].set_title('External probability quality')
sns.histplot(data=shift,x='absolute_smd',hue='dataset',element='step',bins=35,common_norm=False,ax=axes[1])
axes[1].axvline(.5,color='#B22222',ls='--'); axes[1].set_xlabel('Absolute standardized mean difference'); axes[1].set_title('Feature shift relative to CAUEEG')
save_figure(fig,'Figure_4_Probability_Quality_and_Feature_Shift'); plt.show()

## 12. Reviewer-response evidence map

In [ ]:
reviewer_map=pd.DataFrame([
 {'review_issue':'Diagnostic mapping, exclusions, and denominators','evidence':'Authenticated cohort progression; AD-specific harmonization; mutually exclusive exclusions; 914/979 retention','manuscript_location':'Methods—cohort construction; Results—cohort and QA','status':'Addressed'},
 {'review_issue':'Potential diagnosis-related automated-QA exclusion','evidence':'Retention by diagnosis; chi-square/Cramer V; Holm-adjusted pairwise differences; ±10% threshold sensitivity','manuscript_location':'Methods—automated QA and sensitivity; Results—QA audit; Discussion—QA limitation','status':'Addressed; blinded specialist QA remains a limitation'},
 {'review_issue':'Incomplete endpoint scope','evidence':'Normal vs AD, MCI vs AD, Normal vs MCI, and three-class endpoints maintained internally','manuscript_location':'Methods—problem formulation; Results—development comparison','status':'Addressed'},
 {'review_issue':'Fair comparison across methods','evidence':'Unified nested OOF table; controlled representation/classifier audit; method names used in comparisons','manuscript_location':'Methods—model comparison; Results—development benchmark','status':'Addressed'},
 {'review_issue':'Age confounding','evidence':'Fold-safe age-only, EEG, EEG-plus-age, regional, and age-normative analyses with paired deltas','manuscript_location':'Methods—confounding analysis; Results—age sensitivity; Discussion—residual confounding','status':'Addressed within available age data'},
 {'review_issue':'External compatibility and endpoint availability','evidence':'AHEPA restricted to Normal vs AD; Dryad evaluated on four compatible endpoints; participant-level units','manuscript_location':'Methods—external evaluation; Results—transportability','status':'Addressed'},
 {'review_issue':'Calibration and probability quality','evidence':'Brier score, prevalence-reference Brier, Brier skill, log loss, ECE, reliability bins with Wilson intervals','manuscript_location':'Methods—statistical analysis; Results—external probability quality','status':'Addressed'},
 {'review_issue':'Dataset shift and external degradation','evidence':'646-feature SMD distributions and cross-validated domain ROC-AUC; external probability degradation','manuscript_location':'Results—dataset shift; Discussion—transportability','status':'Addressed descriptively; cause not claimed'},
 {'review_issue':'Historically untouched external claim','evidence':'External evidence explicitly labelled locked re-evaluation; no refit, recalibration, threshold optimization, or adaptation','manuscript_location':'Methods and Discussion—external evidence boundary','status':'Addressed'},
 {'review_issue':'Reproducibility and failure controls','evidence':'SHA-256 inventories, passed upstream locks, schema checks, endpoint checks, regeneration manifest','manuscript_location':'Methods—provenance locking; Data/code availability','status':'Addressed'}])
save_table(reviewer_map,'Table_6_Reviewer_Response_Evidence_Map',True); display(reviewer_map)

## 13. Explicit failure-control audit

In [ ]:
failure_controls=pd.DataFrame([
 {'failure_condition':'Missing mandatory final table','enforcement':'File existence check before loading','expected_action':'STOP'},
 {'failure_condition':'Upstream integrity check not passed','enforcement':'All Boolean lock rows must be true','expected_action':'STOP'},
 {'failure_condition':'Endpoint set inconsistent','enforcement':'Exact four-endpoint internal contract and dataset-specific external contract','expected_action':'STOP'},
 {'failure_condition':'QA denominator or retained count changes','enforcement':'Exact 979 and 914 reconciliation','expected_action':'STOP'},
 {'failure_condition':'Feature schema/order mismatch','enforcement':'Inherited H2A ordered 646-feature authentication','expected_action':'STOP upstream'},
 {'failure_condition':'External cohort pooled across datasets','enforcement':'Dataset field retained in every table and panel','expected_action':'PROHIBITED'},
 {'failure_condition':'External tuning or recalibration','enforcement':'No estimator fitting code; imported predictions/metrics only','expected_action':'PROHIBITED'},
 {'failure_condition':'Historically untouched claim','enforcement':'Evidence label fixed as locked external re-evaluation','expected_action':'PROHIBITED'},
 {'failure_condition':'Performance target used as reporting gate','enforcement':'All endpoints and negative results retained','expected_action':'PROHIBITED'}])
save_table(failure_controls,'04_FAILURE_CONTROL_CONTRACT'); display(failure_controls)

## 14. Final publication and reproducibility lock

In [ ]:
final_checks=pd.DataFrame([
 {'check':'all upstream final locks passed','pass':locks.all_passed.all()},
 {'check':'QA exact counts retained','pass':int(retention.montage_eligible_n.sum())==979 and int(retention.retained_n.sum())==914},
 {'check':'four internal endpoints represented','pass':set(dev.task)==set(ENDPOINTS)},
 {'check':'AHEPA endpoint contract respected','pass':set(external.loc[external.dataset.eq('AHEPA'),'task'])=={'normal_vs_ad'}},
 {'check':'Dryad four-endpoint contract respected','pass':set(external.loc[external.dataset.eq('Dryad'),'task'])==set(ENDPOINTS)},
 {'check':'age sensitivity included','pass':len(age)>0 and len(age_deltas)>0},
 {'check':'calibration and Brier skill included','pass':external[['multiclass_brier','prevalence_baseline_brier','brier_skill_score']].notna().all().all()},
 {'check':'feature and domain shift included','pass':len(shift)>0 and len(domain)==2},
 {'check':'reviewer evidence map complete','pass':len(reviewer_map)>=10},
 {'check':'PDF and PNG publication figures regenerated','pass':len(list(MANUSCRIPT.glob('*.pdf')))>=4 and len(list(MANUSCRIPT.glob('*.png')))>=4},
 {'check':'no model fitting, selection, threshold tuning, recalibration, or adaptation','pass':True},
 {'check':'all negative external results retained','pass':len(external)==10},
 {'check':'no performance target used as gate','pass':True}])
save_table(final_checks,'05_FINAL_INTEGRITY_CHECKS'); display(final_checks)
if not final_checks['pass'].all(): raise AssertionError('P1-E16.7H-3 final integrity failure.')

manifest=[]
for path in sorted(OUTPUT_ROOT.rglob('*')):
    if path.is_file() and path.name!='06_OUTPUT_MANIFEST.csv':
        manifest.append({'relative_path':str(path.relative_to(OUTPUT_ROOT)),'bytes':path.stat().st_size,'sha256':sha256_file(path)})
manifest=pd.DataFrame(manifest); save_table(manifest,'06_OUTPUT_MANIFEST')
run={'experiment':'P1-E16.7H-3 final regeneration, failure-control, reviewer-evidence, and publication lock',
     'status':'PASSED','completed_utc':datetime.now(timezone.utc).isoformat(),'endpoints':ENDPOINTS,
     'evidence_layers':['development OOF','retrospective locked official test','locked external re-evaluation'],
     'external_contract':{'AHEPA':['normal_vs_ad'],'Dryad':ENDPOINTS},
     'model_fitting':False,'model_selection':False,'threshold_tuning':False,'recalibration':False,'adaptation':False,
     'historically_untouched_external_claim':False,'performance_target_used':False,'integrity_passed':True,
     'environment':{'python':sys.version,'platform':platform.platform()}}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(run,indent=2),encoding='utf-8')
print('P1-E16.7H-3 PASSED — FINAL REGENERATION, AND PUBLICATION LOCK COMPLETE')